# Natural Gas Price Analysis and Forecasting

This notebook estimates natural gas prices for any requested date by combining historical monthly prices with a 12-month Holt-Winters forecast.


## 1. Import Libraries

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from statsmodels.tsa.holtwinters import ExponentialSmoothing

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Load and Prepare the Data

In [ ]:
local_path = Path('Nat_Gas.csv')
colab_path = Path('/content/Nat_Gas.csv')

if local_path.exists():
    csv_path = local_path
elif colab_path.exists():
    csv_path = colab_path
else:
    raise FileNotFoundError(
        "Nat_Gas.csv was not found. Put it beside this notebook or upload it to /content/Nat_Gas.csv."
    )

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

df['Dates'] = pd.to_datetime(df['Dates'])
df['Prices'] = pd.to_numeric(df['Prices'], errors='coerce')
df = df.dropna(subset=['Dates', 'Prices']).sort_values('Dates')
df = df.set_index('Dates')

df.head()

## 3. Visualize Historical Prices

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['Prices'], marker='o', linewidth=2)
ax.set_title('Historical Natural Gas Prices')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
plt.show()

## 4. Build the Forecast Model

The data is monthly, so the seasonal period is set to 12.

In [ ]:
model = ExponentialSmoothing(
    df['Prices'],
    trend='add',
    seasonal='add',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

forecast_steps = 12
forecast = model.forecast(forecast_steps)
future_df = forecast.to_frame(name='Prices')

combined_df = pd.concat([df, future_df])
future_df

## 5. Plot Historical and Forecasted Prices

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df['Prices'], label='Historical Prices', marker='o', linewidth=2)
ax.plot(future_df.index, future_df['Prices'], label='Forecasted Prices', marker='o', linestyle='--', linewidth=2)
ax.set_title('Natural Gas Prices: Historical and Forecasted')
ax.set_xlabel('Date')
ax.set_ylabel('Price')
ax.legend()
plt.show()

## 6. Estimate Price for Any Date

The function below converts dates into numeric values and uses linear interpolation between known monthly values. Dates after the forecast range are extrapolated.

In [ ]:
x = combined_df.index.map(pd.Timestamp.toordinal).to_numpy()
y = combined_df['Prices'].to_numpy()

price_function = interp1d(x, y, kind='linear', fill_value='extrapolate')

def estimate_price(date_string):
    """Return the estimated natural gas price for a date.

    Parameters
    ----------
    date_string : str
        Date in a format accepted by pandas, for example '2025-03-15'.

    Returns
    -------
    float
        Estimated gas price rounded to two decimal places.
    """
    date = pd.to_datetime(date_string)
    estimated_price = float(price_function(date.toordinal()))
    return round(estimated_price, 2)

## 7. Test the Function

In [ ]:
test_dates = ['2025-03-15', '2023-07-10', '2024-12-31']

for test_date in test_dates:
    print(f"Estimated price on {test_date}: {estimate_price(test_date)}")